In [0]:
-- In Free Edition, we use the default hive_metastore
USE CATALOG `prism-sentinel-stream`;

-- Create your Medallion layers as Schemas (Databases)
CREATE SCHEMA IF NOT EXISTS prism_bronze;
CREATE SCHEMA IF NOT EXISTS prism_silver;
CREATE SCHEMA IF NOT EXISTS prism_gold;

-- Verify they are created
SHOW SCHEMAS;

In [0]:
-- 1. Create the Silver Schema
CREATE SCHEMA IF NOT EXISTS `prism-sentinel-stream`.prism_silver;

-- 2. Create the Sanctions Master Table with History Tracking
CREATE TABLE IF NOT EXISTS `prism-sentinel-stream`.prism_silver.sanctions_master (
    entity_id STRING,
    entity_name STRING,
    risk_level STRING,
    is_current BOOLEAN,
    valid_from TIMESTAMP,
    valid_to TIMESTAMP
) USING DELTA;

-- 3. Seed with Initial "High Risk" Entities
INSERT INTO `prism-sentinel-stream`.prism_silver.sanctions_master
VALUES 
('ENT_001', 'NORTH_STAR_SHIPPING', 'CRITICAL', true, current_timestamp(), NULL),
('ENT_002', 'GLOBAL_LOGISTICS', 'HIGH', true, current_timestamp(), NULL),
('ENT_003', 'SHELL_CO', 'MEDIUM', true, current_timestamp(), NULL);

In [0]:
%sql
SELECT 
    counterparty, 
    entity_name as matched_entity, 
    risk_level,
    count(*) as alert_count
FROM `prism-sentinel-stream`.prism_silver.transactions_refined 
WHERE risk_flag = 'ALERT' 
GROUP BY 1, 2, 3
ORDER BY alert_count DESC;

In [0]:
SELECT 
    country, 
    total_value 
FROM `prism-sentinel-stream`.prism_gold.daily_risk_summary
WHERE risk_flag = 'ALERT';